# 1. Gaussian amplitudes and simulated samples


    We construct a positive interference model with known densities and derivatives.
    This provides an analytical benchmark for learned density ratios, unbinned
    likelihoods, and parameterized score histograms. Run this notebook first.

    Five million generated events per sample provide Monte Carlo precision; the
    expected event yields of the statistical experiment are separate quantities.

### Runtime and persistent run

In Colab, choose **Runtime → Change runtime type → GPU** for notebooks 2–3.
The source checkout lives in the temporary runtime; **datasets, checkpoints,
workspace files, and results live in Google Drive**. Use the same `RUN_NAME`
and `MODE` in every notebook. Set `MODE = "smoke"` for a short workflow check;
the production default generates five million events **per sample**.
Use `CONFIG_OVERRIDES` to change a computational budget without editing source,
for example `{"quadrature_per_process": 500_000, "epochs": 150}`. Use the same
overrides in all five notebooks, and choose a new `RUN_NAME` when changing them.

This repository may be private. Grant Colab access when opening it from GitHub.
For the runtime download, either save a GitHub fine-grained token with **Contents:
read** access to this repository as the Colab secret `GITHUB_TOKEN`, or enter it
in the hidden prompt. The token is used only in a Python HTTP header and is not
written into files, Git remotes, shell arguments, or notebook output. Alternatively,
upload a repository ZIP to `/content/poodemo_source.zip`; no token is then needed.
An existing `/content/poodemo` checkout is reused. Delete that checkout to fetch
new source after updating the repository; your Drive run is separate.
For this NI-only revision, use the new default `RUN_NAME = "demo-ni-v1"`.
If you ran an earlier version, start a fresh Colab runtime (or refresh the
temporary source checkout and restart the runtime) before proceeding. Existing
run manifests from the earlier model are incompatible and are not reused.

In [ ]:
import os
import sys
from pathlib import Path

RUN_NAME = "demo-ni-v1"
MODE = os.environ.get("POODEMO_MODE", "production")  # production or smoke
CONFIG_OVERRIDES = {}  # e.g. {"quadrature_per_process": 500_000, "epochs": 150}

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path(os.environ.get("POODEMO_ROOT", f"/content/drive/MyDrive/poodemo/runs/{RUN_NAME}"))
    REPO_DIR = Path("/content/poodemo")
    if not (REPO_DIR / "pyproject.toml").exists():
        import getpass
        import io
        import shutil
        import tarfile
        import tempfile
        import urllib.error
        import urllib.request
        import zipfile

        uploaded_zip = Path("/content/poodemo_source.zip")
        if uploaded_zip.exists():
            archive_bytes = uploaded_zip.read_bytes()
            archive_kind = "zip"
        else:
            archive_url = "https://api.github.com/repos/rafaellopesdesa/poodemo/tarball/main"

            def fetch_source(token=None):
                headers = {"Accept": "application/vnd.github+json", "User-Agent": "poodemo-colab"}
                if token:
                    headers["Authorization"] = "Bearer " + token
                request = urllib.request.Request(archive_url, headers=headers)
                with urllib.request.urlopen(request, timeout=180) as response:
                    return response.read()

            try:
                archive_bytes = fetch_source()
            except urllib.error.HTTPError as error:
                if error.code not in (401, 403, 404):
                    raise RuntimeError(f"Repository download failed (HTTP {error.code}).") from None
                token = None
                try:
                    from google.colab import userdata
                    token = userdata.get("GITHUB_TOKEN")
                except Exception:
                    pass
                if not token:
                    token = getpass.getpass("GitHub token with read access to poodemo: ").strip()
                if not token:
                    raise RuntimeError("Upload /content/poodemo_source.zip or provide read access.")
                try:
                    archive_bytes = fetch_source(token)
                except urllib.error.HTTPError as auth_error:
                    raise RuntimeError(f"Repository download failed (HTTP {auth_error.code}); check token access.") from None
                finally:
                    token = None
            archive_kind = "tar"

        # Validate archive paths and reject links before extracting the source.
        with tempfile.TemporaryDirectory(prefix="poodemo-source-") as temp:
            staging = Path(temp).resolve()

            def safe_destination(name):
                destination = (staging / name).resolve()
                if not destination.is_relative_to(staging):
                    raise RuntimeError("Unsafe path in source archive.")
                return destination

            if archive_kind == "zip":
                with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
                    for entry in archive.infolist():
                        safe_destination(entry.filename)
                        if (entry.external_attr >> 16) & 0o170000 == 0o120000:
                            raise RuntimeError("Symbolic links are not supported in the source ZIP.")
                    archive.extractall(staging)
            else:
                with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
                    for entry in archive.getmembers():
                        safe_destination(entry.name)
                        if not (entry.isfile() or entry.isdir()):
                            raise RuntimeError("Links and special files are not supported in the source archive.")
                    archive.extractall(staging)
            candidates = [p.parent for p in staging.rglob("pyproject.toml") if (p.parent / "poodemo").is_dir()]
            if len(candidates) != 1:
                raise RuntimeError("The uploaded archive must contain one poodemo repository checkout.")
            shutil.copytree(candidates[0], REPO_DIR, dirs_exist_ok=True)
        del archive_bytes

    import subprocess
    if os.environ.get("POODEMO_SKIP_INSTALL") != "1":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-colab.txt")])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
else:
    # Locally: pip install -r requirements-colab.txt && pip install -e .
    ROOT = Path(os.environ.get("POODEMO_ROOT", str(Path.home() / "poodemo-runs" / RUN_NAME)))
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "poodemo" / "pipeline.py").exists():
            sys.path.insert(0, str(candidate))
            break

ROOT.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {ROOT}")
print(f"Mode: {MODE}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poodemo.pipeline import create_run

run = create_run(ROOT, mode=MODE, overrides=CONFIG_OVERRIDES)
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "grid.alpha": 0.2})
display(pd.Series(run.config, name="configuration"))

### Amplitudes, densities, and yields

Let \(\phi_j(x)=\mathcal N_3(x;m_j,\Sigma_j)\), normalized to one.
Define amplitudes
\[
A_S(x)=\sqrt{\lambda_S\phi_S(x)},\qquad
A_B(x)=e^{i\varphi}\sqrt{\lambda_B\phi_B(x)},
\qquad A_{NI}(x)=\sqrt{\lambda_{NI}\phi_{NI}(x)}.
\]
Their square roots are Gaussian functions, so these are Gaussian
wavefunctions. We choose \(\cos\varphi=-0.65\), distinct means and
positive-definite covariance matrices, and inclusive yields
\(\lambda_S=100,\lambda_B=1000,\lambda_{NI}=10000\).

The coherent process at \(\mu=1\) is
\[
D_{SBI}=|A_S+A_B|^2=D_S+D_B+D_I,
\quad D_I=2\cos\varphi\sqrt{D_SD_B}.
\]
The SBI density is \(p_{SBI}=D_{SBI}/\lambda_{SBI}\), with
\(\lambda_{SBI}=\int D_{SBI}\,dx\). It is **not** assigned an arbitrary
yield: interference fixes its normalization. NI is added incoherently.

At any \(\mu\ge0\),
\[
D(x;\mu)=|\sqrt\mu A_S+A_B|^2+D_{NI}
= (\mu-\sqrt\mu)D_S+\sqrt\mu D_{SBI}
  +(1-\sqrt\mu)D_B+D_{NI}.
\]
The negative coefficients in this basis are physical algebra; the total
amplitude model remains positive.

### Shape uncertainties and statistically independent roles

The only nuisance is \(\alpha_{NI}\). A reproducibly chosen unit direction
\(v_{NI}\) shifts the NI amplitude mean by
\(\Delta m_{NI}=0.1\|m_{NI}\|v_{NI}\) for one standard deviation.
S, B, and their coherent SBI process are fixed with respect to this
nuisance. The exact NI shift vector and all model settings are saved in
the run configuration. The inclusive NI yield stays fixed; its selected
yield may change because the selection efficiency changes.

This gives six samples: S, B, SBI, NI, NI_up, and NI_down. Production mode
therefore generates 30 million three-dimensional events. Before any
learning or selection, each sample is divided into seven independent
roles:

| Role | Fraction |
|---|---:|
| Preselection training | 20% |
| Preselection validation | 5% |
| Selection threshold calibration | 5% |
| Density-ratio training | 40% |
| Density-ratio validation | 10% |
| Density-ratio calibration | 5% |
| Final integration | 15% |

Events used to train or tune the selector are never reused for the
subsequent ratio-training or inference stages. A sample's Monte Carlo
size does not change its physical yield.

In [ ]:
display(pd.Series(run.model.to_dict(), name="amplitude model"))

In [ ]:
from poodemo.pipeline import generate_data

sample_summary = generate_data(run)
display(sample_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
edges = np.linspace(-4, 5, 75)
for process in ["S", "SBI", "B", "NI"]:
    points = np.load(ROOT / "raw" / f"{process}.npy", mmap_mode="r")[:100_000]
    for coordinate, ax in enumerate(axes):
        ax.hist(points[:, coordinate], bins=edges, density=True, histtype="step", label=process)
        ax.set(xlabel=fr"$x_{coordinate+1}$", ylabel="Normalized marginal density")
axes[0].legend()
fig.tight_layout()
(ROOT / "plots").mkdir(exist_ok=True)
fig.savefig(ROOT / "plots" / "01_process_marginals.pdf", bbox_inches="tight")
plt.show()

### Inspect the saved experiment

The generator writes samples in chunks and records normalization and
sampling information. Re-running a completed stage reuses its saved
products; changing the experiment should use a new `RUN_NAME`.
Selected-data and quadrature caches are tied to the selector's
fingerprint, preventing reuse after the selection model changes.
Inspect the summary above for the generated counts and physical yields.
The selected yields in notebook 2 will differ because the classifier cut
has different efficiencies for each process and each systematic anchor.

In [ ]:
print("Persistent products:")
for path in sorted(ROOT.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))

Next: open **02_preselection.ipynb** with the same run name and mode.